# Notebook 4: Audio Feature Extraction + Classifier

**Account:** B | **GPU:** T4 x2 | **Time:** ~8-10 hrs

**Datasets to attach:** `daisee-videos` + `smartlms-scripts`

**Step 1:** Extract prosodic features (pitch, energy, pauses) from video audio
**Step 2:** Extract wav2vec2 learned embeddings
**Step 3:** Train AudioMLP classifier on combined audio features

DAiSEE videos contain audio that is COMPLETELY UNTAPPED in existing work —
this is a novel contribution for the paper.

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q "numpy<2" librosa soundfile transformers xgboost optuna
# librosa needs numba which Kaggle already has

import torch, os, sys, shutil, glob, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"Multi-GPU: {torch.cuda.device_count()} GPUs — DataParallel enabled")

# Check ffmpeg (needed for audio extraction)
result = subprocess.run(["which", "ffmpeg"], capture_output=True, text=True)
print(f"ffmpeg: {result.stdout.strip() or 'NOT FOUND — install with: !apt-get install -y ffmpeg'}")

In [ ]:
# ── Cell 2: Copy scripts + setup ──
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/app/ml", exist_ok=True)
open(f"{WORK}/app/__init__.py", "w").close()
open(f"{WORK}/app/ml/__init__.py", "w").close()

def find_script(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

scripts = [
    "train_model_v2.py", "train_model_v3.py", "train_model_v4.py",
    "train_multimodal_v5.py", "extract_face_embeddings.py",
    "train_videomae.py", "extract_audio_features.py"
]
for s in scripts:
    found = find_script(s)
    if found:
        shutil.copy(found, f"{WORK}/app/ml/{s}")
        print(f"✓ {s} ← {found}")
    else:
        print(f"✗ NOT FOUND: {s}")

sys.path.insert(0, WORK)
os.chdir(WORK)

In [ ]:
# ── Cell 3: Auto-detect & patch paths ──
print("Attached datasets:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  /kaggle/input/{d}/")

# Find video DataSet directory
result = subprocess.run(["find", "/kaggle/input", "-name", "DataSet", "-type", "d"], capture_output=True, text=True)
dataset_dirs = [l for l in result.stdout.strip().split("\n") if l]
VIDEO_DIR = dataset_dirs[0] if dataset_dirs else None
print(f"\nVideo DataSet dir: {VIDEO_DIR}")

# Count videos
if VIDEO_DIR:
    n_avi = len(glob.glob(os.path.join(VIDEO_DIR, "**", "*.avi"), recursive=True))
    print(f"Found {n_avi} .avi video clips")

# Patch extract_audio_features.py
audio_path = f"{WORK}/app/ml/extract_audio_features.py"
code = open(audio_path).read()
if VIDEO_DIR:
    code = code.replace(
        'VIDEO_DIR = "/kaggle/input/daisee-videos/DAiSEE/DataSet"',
        f'VIDEO_DIR = "{VIDEO_DIR}"'
    )
open(audio_path, 'w').write(code)
print("\n✓ Patched extract_audio_features.py")

In [ ]:
# ── Cell 4: Extract prosodic + wav2vec2 features (~6-8 hrs) ──
# Processes all ~8231 video clips, skips already-processed clips
from app.ml.extract_audio_features import extract_all_audio_features
import torch, gc

extract_all_audio_features(mode="both", resume=True)

gc.collect()
torch.cuda.empty_cache()
print("\n✅ Audio feature extraction complete!")

In [ ]:
# ── Cell 5: Check extracted features ──
import numpy as np

audio_dir = "/kaggle/working/audio_features"
prosodic = glob.glob(os.path.join(audio_dir, "prosodic", "*.npy"))
wav2vec = glob.glob(os.path.join(audio_dir, "wav2vec", "*.npy"))
print(f"Prosodic features: {len(prosodic)} clips")
print(f"wav2vec features:  {len(wav2vec)} clips")

if prosodic:
    sample = np.load(prosodic[0])
    print(f"\nProsodic sample shape: {sample.shape}")
if wav2vec:
    sample = np.load(wav2vec[0])
    print(f"wav2vec sample shape:  {sample.shape}")

In [ ]:
# ── Cell 6: Train AudioMLP classifier (~2 hrs) ──
from app.ml.extract_audio_features import train_audio_classifier
import torch, gc

for dim in ["boredom", "engagement", "confusion", "frustration"]:
    print(f"\n{'='*60}")
    print(f"Training audio classifier for: {dim}")
    print(f"{'='*60}")
    train_audio_classifier(dim, "both")
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ Audio classifier training complete!")

In [ ]:
# ── Cell 7: Review results ──
import json

print("=" * 80)
print("AUDIO CLASSIFIER RESULTS")
print("=" * 80)

for rf in sorted(glob.glob("/kaggle/working/experiment_results/*audio*.json")):
    print(f"\n{os.path.basename(rf)}")
    with open(rf) as f:
        data = json.load(f)
    for key, val in data.items():
        if isinstance(val, dict) and ('test_f1_macro' in val or 'f1_macro' in val):
            f1 = val.get('test_f1_macro') or val.get('f1_macro', 0)
            print(f"  {key}: F1m = {f1:.4f}")
        elif isinstance(val, (int, float)):
            print(f"  {key}: {val}")

In [ ]:
# ── Cell 8: Save outputs ──
shutil.make_archive("/kaggle/working/audio_features_zip", 'zip', "/kaggle/working/audio_features")
shutil.make_archive("/kaggle/working/audio_results", 'zip', "/kaggle/working/experiment_results")

for f in glob.glob("/kaggle/working/*.zip"):
    size_mb = os.path.getsize(f) / 1e6
    print(f"{os.path.basename(f)}: {size_mb:.1f} MB")

print("\n✅ Download from Output tab!")
print("The audio_features/ folder will be needed for Notebook 5 (Fusion).")
print("TIP: Re-upload 'audio_features' as a new Kaggle dataset for Notebook 5.")